# 马尔可夫决策过程原文摘要

来源文章： [动手学强化学习 - 马尔可夫决策过程](https://hrl.boyuai.com/chapter/1/%E9%A9%AC%E5%B0%94%E5%8F%AF%E5%A4%AB%E5%86%B3%E7%AD%96%E8%BF%87%E7%A8%8B)

这份 notebook 以原文摘要为主，同时补入几段文章中的关键示例代码，方便一边看概念，一边对照实现。


## 1. 原文主线

文章按下面这条线展开：

- 马尔可夫过程（MP）
- 马尔可夫奖励过程（MRP）
- 马尔可夫决策过程（MDP）
- 贝尔曼方程
- Monte Carlo 估计
- 占用度量
- 最优策略与贝尔曼最优方程

核心目标是先把强化学习里最基础的环境描述和价值函数体系搭起来。


## 2. MRP：回报与解析解

原文先从 MRP 入手。MRP 可以写成 `(S, P, R, \gamma)`：

- `S` 是状态集合
- `P` 是状态转移矩阵
- `R` 是奖励函数
- `\gamma` 是折扣因子

从某个状态开始的回报是未来奖励的折扣和，状态价值函数满足贝尔曼方程，因此可以用矩阵形式直接求解析解。

下面这段代码对应原文 3.3 节的示例：先定义 MRP，再计算一条状态链的回报和所有状态的价值。


In [3]:
import numpy as np

np.random.seed(0)

# 定义状态转移概率矩阵P
P = [
    [0.9, 0.1, 0.0, 0.0, 0.0, 0.0],
    [0.5, 0.0, 0.5, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.6, 0.0, 0.4],
    [0.0, 0.0, 0.0, 0.0, 0.3, 0.7],
    [0.0, 0.2, 0.3, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
]
P = np.array(P)

rewards = [-1, -2, -2, 10, 1, 0]
gamma = 0.5


def compute_return(start_index, chain, gamma):
    G = 0
    for i in reversed(range(start_index, len(chain))):
        G = gamma * G + rewards[chain[i] - 1]
    return G


def compute(P, rewards, gamma, states_num):
    rewards = np.array(rewards).reshape((-1, 1))
    value = np.dot(np.linalg.inv(np.eye(states_num, states_num) - gamma * P), rewards)
    return value


chain = [1, 2, 3, 6]
G = compute_return(0, chain, gamma)
print("根据本序列计算得到回报为：%s。" % G)

V_mrp = compute(P, rewards, gamma, 6)
print("MRP中每个状态价值分别为\n", V_mrp)


根据本序列计算得到回报为：-2.5。
MRP中每个状态价值分别为
 [[-2.01950168]
 [-2.21451846]
 [ 1.16142785]
 [10.53809283]
 [ 3.58728554]
 [ 0.        ]]


## 3. MDP：状态、动作、策略

MDP 比 MRP 多了动作，因此写成 `(S, A, P, R, \gamma)`。

原文这里给了一个 5 状态的玩具 MDP，并定义了两个策略：

- `Pi_1`：随机策略
- `Pi_2`：另一组偏向不同动作的策略

下面这段就是原文里后续 Monte Carlo 与占用度量都复用的基础环境定义。


In [4]:
S = ["s1", "s2", "s3", "s4", "s5"]
A = ["保持s1", "前往s1", "前往s2", "前往s3", "前往s4", "前往s5", "概率前往"]

P = {
    "s1-保持s1-s1": 1.0,
    "s1-前往s2-s2": 1.0,
    "s2-前往s1-s1": 1.0,
    "s2-前往s3-s3": 1.0,
    "s3-前往s4-s4": 1.0,
    "s3-前往s5-s5": 1.0,
    "s4-前往s5-s5": 1.0,
    "s4-概率前往-s2": 0.2,
    "s4-概率前往-s3": 0.4,
    "s4-概率前往-s4": 0.4,
}

R = {
    "s1-保持s1": -1,
    "s1-前往s2": 0,
    "s2-前往s1": -1,
    "s2-前往s3": -2,
    "s3-前往s4": -2,
    "s3-前往s5": 0,
    "s4-前往s5": 10,
    "s4-概率前往": 1,
}

gamma = 0.5
MDP = (S, A, P, R, gamma)

Pi_1 = {
    "s1-保持s1": 0.5,
    "s1-前往s2": 0.5,
    "s2-前往s1": 0.5,
    "s2-前往s3": 0.5,
    "s3-前往s4": 0.5,
    "s3-前往s5": 0.5,
    "s4-前往s5": 0.5,
    "s4-概率前往": 0.5,
}

Pi_2 = {
    "s1-保持s1": 0.6,
    "s1-前往s2": 0.4,
    "s2-前往s1": 0.3,
    "s2-前往s3": 0.7,
    "s3-前往s4": 0.5,
    "s3-前往s5": 0.5,
    "s4-前往s5": 0.1,
    "s4-概率前往": 0.9,
}


def join(str1, str2):
    return str1 + '-' + str2


## 4. 固定策略后把 MDP 化成 MRP

原文的重要结论是：给定一个策略以后，可以把动作边缘化，把 MDP 转成 MRP。这样就能继续使用前面的解析解方法。

原文为了简洁，直接给出了 `Pi_1` 对应的转化结果：新的状态转移矩阵和奖励向量如下。然后用前面定义好的 `compute` 计算状态价值。


In [5]:
gamma = 0.5

P_from_mdp_to_mrp = [
    [0.5, 0.5, 0.0, 0.0, 0.0],
    [0.5, 0.0, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.5, 0.5],
    [0.0, 0.1, 0.2, 0.2, 0.5],
    [0.0, 0.0, 0.0, 0.0, 1.0],
]
P_from_mdp_to_mrp = np.array(P_from_mdp_to_mrp)
R_from_mdp_to_mrp = [-0.5, -1.5, -1.0, 5.5, 0]

V_pi_1 = compute(P_from_mdp_to_mrp, R_from_mdp_to_mrp, gamma, 5)
print("MDP中每个状态价值分别为\n", V_pi_1)


MDP中每个状态价值分别为
 [[-1.22555411]
 [-1.67666232]
 [ 0.51890482]
 [ 6.0756193 ]
 [ 0.        ]]


## 5. Monte Carlo：靠采样估计状态价值

如果不想显式求解解析解，或者环境模型未知，就可以通过采样 episode 来估计状态价值。原文这里分两步：

- 先定义采样函数 `sample`
- 再定义 `MC`，把每条序列从后往前计算回报，并对每个状态做增量平均

下面两段代码基本就是原文的实现。


In [6]:
def sample(MDP, Pi, timestep_max, number):
    S, A, P, R, gamma = MDP
    episodes = []
    for _ in range(number):
        episode = []
        timestep = 0
        s = S[np.random.randint(4)]
        while s != "s5" and timestep <= timestep_max:
            timestep += 1
            rand, temp = np.random.rand(), 0
            for a_opt in A:
                temp += Pi.get(join(s, a_opt), 0)
                if temp > rand:
                    a = a_opt
                    r = R.get(join(s, a), 0)
                    break
            rand, temp = np.random.rand(), 0
            for s_opt in S:
                temp += P.get(join(join(s, a), s_opt), 0)
                if temp > rand:
                    s_next = s_opt
                    break
            episode.append((s, a, r, s_next))
            s = s_next
        episodes.append(episode)
    return episodes


episodes_preview = sample(MDP, Pi_1, 20, 5)
print("第一条序列\n", episodes_preview[0])
print("第二条序列\n", episodes_preview[1])
print("第五条序列\n", episodes_preview[4])


第一条序列
 [('s1', '前往s2', 0, 's2'), ('s2', '前往s3', -2, 's3'), ('s3', '前往s5', 0, 's5')]
第二条序列
 [('s4', '概率前往', 1, 's4'), ('s4', '前往s5', 10, 's5')]
第五条序列
 [('s2', '前往s3', -2, 's3'), ('s3', '前往s4', -2, 's4'), ('s4', '前往s5', 10, 's5')]


In [7]:
def MC(episodes, V, N, gamma):
    for episode in episodes:
        G = 0
        for i in range(len(episode) - 1, -1, -1):
            (s, a, r, s_next) = episode[i]
            G = r + gamma * G
            N[s] = N[s] + 1
            V[s] = V[s] + (G - V[s]) / N[s]


timestep_max = 20
episodes = sample(MDP, Pi_1, timestep_max, 1000)
gamma = 0.5
V = {"s1": 0, "s2": 0, "s3": 0, "s4": 0, "s5": 0}
N = {"s1": 0, "s2": 0, "s3": 0, "s4": 0, "s5": 0}
MC(episodes, V, N, gamma)
print("使用蒙特卡洛方法计算MDP的状态价值为\n", V)


使用蒙特卡洛方法计算MDP的状态价值为
 {'s1': -1.228923788722258, 's2': -1.6955696284402704, 's3': 0.4823809701532294, 's4': 5.967514743019431, 's5': 0}


## 6. 占用度量

原文在这一节解释：同一个 MDP 下，不同策略会访问到不同的状态和状态动作对，所以它们的价值函数也会不同。为了刻画这种差异，引入了占用度量。

直观上，它衡量的是某个状态动作对 `(s, a)` 在策略下被访问到的折扣频率。原文最后给出一段近似估计代码，通过大量采样来计算占用度量。


In [8]:
def occupancy(episodes, s, a, timestep_max, gamma):
    rho = 0
    total_times = np.zeros(timestep_max)
    occur_times = np.zeros(timestep_max)
    for episode in episodes:
        for i in range(len(episode)):
            (s_opt, a_opt, r, s_next) = episode[i]
            total_times[i] += 1
            if s == s_opt and a == a_opt:
                occur_times[i] += 1
    for i in reversed(range(timestep_max)):
        if total_times[i]:
            rho += gamma**i * occur_times[i] / total_times[i]
    return (1 - gamma) * rho


gamma = 0.5
timestep_max = 1000
episodes_1 = sample(MDP, Pi_1, timestep_max, 1000)
episodes_2 = sample(MDP, Pi_2, timestep_max, 1000)
rho_1 = occupancy(episodes_1, "s4", "概率前往", timestep_max, gamma)
rho_2 = occupancy(episodes_2, "s4", "概率前往", timestep_max, gamma)
print(rho_1, rho_2)


0.112567796310472 0.23199480615618912


## 7. 最优策略

文章最后从“评价一个给定策略”转向“寻找最优策略”。

- 最优状态价值函数是所有策略下状态价值的最大值
- 最优动作价值函数是所有策略下动作价值的最大值
- 它们满足贝尔曼最优方程

这一节的作用是把前面的价值函数概念，衔接到后续动态规划、时序差分和 Q-learning 这些真正求最优策略的方法上。


## 8. 一页总结

- MP 描述状态如何随机变化
- MRP 在此基础上加入奖励和回报
- MDP 再加入动作和策略
- 贝尔曼方程给出价值函数的递推结构
- Monte Carlo 用采样近似价值函数
- 占用度量描述策略访问状态动作对的频率
- 最优策略问题是后续强化学习算法的核心目标

这一章的真正意义，是先把强化学习后续章节会反复用到的语言体系搭起来。
